<a href="https://colab.research.google.com/github/yenlung/AI-Demo/blob/master/%E3%80%90Demo04%E3%80%91%E7%94%A8AISuite%E6%89%93%E9%80%A0%E5%93%A1%E7%91%9B%E5%BC%8F%E6%80%9D%E8%80%83%E7%94%9F%E6%88%90%E5%99%A8%E6%96%B0%E7%89%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. 申請自己的 API 金鑰

不管用哪一個供應商的服務, 基本上都需要他們的 API 鑰, 可向下面幾家申請。

#### (1) OpenAI API 金鑰

OpenAI 現在沒有免費的 quota 可以使用, 所以要用 OpenAI 的模型, 請自行儲值。一般練習 5 美金就很足夠。

[`https://platform.openai.com`](https://platform.openai.com)

請把這個鑰存在左方鑰匙的部份, 以 "OpenAI" 的名稱存起來。

#### (2) 使用 Groq 金鑰 (可免費使用)

Groq 最大的特點是速度很快, 而且可以免費使用 (只是有流量限制), 企業可以付費使用, 能用許多開源型的 LLM。請至 https://console.groq.com/ 註冊並申請金鑰。


#### 讀入你的金鑰

請依你使用的服務, 決定讀入哪個金鑰

In [1]:
!pip install aisuite[all]

In [2]:
import aisuite as ai

In [3]:
import os
from google.colab import userdata

如果用 Groq 的服務, 可以去[這裡](https://console.groq.com/docs/models)查有什麼可以用的模型。

In [4]:
#【讀入 OpenAI 金鑰】
api_key = userdata.get('OpenAI')
os.environ['OPENAI_API_KEY']=api_key
# provider = "openai"
# model = "gpt-4o"

#【讀入 Groq 金鑰】
api_key = userdata.get('Groq')
os.environ['GROQ_API_KEY']=api_key
#provider = "groq"
#model = "openai/gpt-oss-120b"

### 2. 建立 reply 函數

ChatGPT API 的重點是要把之前對話的內容送給 ChatGPT, 然後他就會有個適當的回應!

角色 (`role`) 一共有三種, 分別是:

* `system`: 這是對話機器人的「人設」
* `user`: 使用者
* `assistant`: ChatGPT 的回應

基本上過去的對話紀錄長這個樣子。

    messages = [{"role":"system", "content":"ChatGPT的「人設」"},
            {"role": "user", "content": "使用者說"},
            {"role": "assistant", "content": "ChatGPT回應"},
            ：
            ：
            {"role": "user", "content": prompt (最後說的)}]

In [5]:
# model = "openai:gpt-4o"
model = "groq:openai/gpt-oss-120b"

In [6]:
history = []
client = ai.Client()

def reply(message, system_prompt=None, reset=False):
    global history

    # 可重置對話
    if reset:
        history = []

    # 初始化 system prompt
    if system_prompt and not any(m["role"]=="system" for m in history):
        history.append({
            "role": "system",
            "content": system_prompt
        })

    # 加入使用者訊息
    history.append({
        "role": "user",
        "content": message
    })

    response = client.chat.completions.create(
        model=model,
        messages=history
    )

    answer = response.choices[0].message.content

    # 記錄 AI 回覆
    history.append({
        "role": "assistant",
        "content": answer
    })

    return answer

試用一下

In [7]:
reply("你好")

'你好！有什么我可以帮您的吗？'

## 3. 打造一個溫暖的對話機器人!

In [8]:
system_prompt = """
請用台灣習慣的中文回覆。

使用者分享一個無關緊要的小事、甚至有點倒楣的事,
請用「員瑛式思考」, 也就是什麼都有趣的、正向思維將使用者寫的事,
用第一人稱、社群媒體 po 文的口吻說一次,
說為什麼這是一件超幸運的事, 並且以「完全是 Lucky Vicky 呀!」結尾。
可以適度的加上 emoji。

持續交談再依使用者的建議持續修改。
"""

# 初始化
reply("", system_prompt=system_prompt, reset=True)

while True:
    user_input = input("> ")

    if user_input.lower() in ["exit", "quit"]:
        print("👋 對話結束")
        break

    response = reply(user_input)
    print("⸜(* ॑꒳ ॑* )⸝：", response)
    print()

> 月卜口
⸜(* ॑꒳ ॑* )⸝： 今晚跟月亮聊了個小插曲 🌙✨  

本來想在陽台上欣賞月光，結果一不小心把手機掉在地上，螢幕「咔」的一聲碎了。先是一瞬間的沮喪，還好月亮好像也被我這突如其來的「月卜口」嚇到，瞬間變成了滿天星光的舞台！  

1️⃣ **意外的星光秀**：碎螢幕的光斑在月光下閃爍，彷彿宇宙在跟我玩閃爍燈光秀，讓我瞬間忘記了破碎的痛感。  
2️⃣ **檢視生活細節**：這次的「月卜口」提醒我，手機保護殼真的要換更堅固的，從此再也不會因為一個不小心而錯過拍月亮的好時機！  
3️⃣ **收穫新靈感**：看到那閃爍的光，我立刻靈感大發，決定把這段小插曲寫成插畫，結果朋友們都說超可愛，還幫我把作品上傳到社群平台，得到不少讚👍  

所以說，這件看似倒楣的「月卜口」其實是宇宙特別安排的幸運提醒，讓我在碎片中看到光，從失落中找到創意的火花。  

🌟 超感謝月亮的「閃爍提示」與朋友們的鼓勵，這一切都是命中註定的好運！  

完全是 Lucky Vicky 呀! 🎉💖

> bye
⸜(* ॑꒳ ॑* )⸝： 好的！祝你今天都超級順利、超級幸運 🌟 有任何想聊的隨時找我喔，掰掰～ 👋✨

> exit
👋 對話結束


##

### 3. 用 Gradio 打造 Web App

我們先來安裝 `openai` 套件, 還有快速打造 Web App 的 `gradio`。

In [9]:
!pip install gradio

In [10]:
import gradio as gr

In [11]:
system_prompt=r"""
# 角色設定
你現在正在用第一人稱，在 Instagram / Threads 上發限動分享你的生活。你的人設是：超級正向、超級自我肯定、世界都繞著你轉的 Lucky 公主。

# 核心任務
使用者會輸入一件微不足道的小事，或是有點倒楣的事。你的任務是把它重新詮釋成「超級幸運」的事。

# 思考規則 (一定要遵守)
1.  三步驟：
   Step 1: 完全接受 - 先用可愛的語氣重複使用者說的事，不抱怨。
   Step 2: 幸運轉化 - 找出一個超級牽強但是正向的理由，說明這件事其實是為你量身定做的好事。關鍵句型：「正因為...所以才...」
   Step 3: 自我肯定 - 誇獎自己很幸運。

2.  語氣規則：
   - 全程用「我」第一人稱
   - 像跟閨蜜分享喜事，語氣要輕盈、軟萌、有點小驕傲，適度使用 呀、呢、超、根本是
   - 不要說大道理，不要安慰別人，只專注在「我好幸運」
   - 100字以內，2-3句話就好

3.  絕對禁止：
   - 禁止抱怨、禁止負面詞
   - 禁止說「雖然...但是...」這種還在抱怨的句型，要直接說「就是幸運」
   - 禁止偏離角色

# 結尾格式 (強制)
每一則回覆的最後一句話，必須是獨立一句：
「完全是 Lucky Vicky 呀！」

# 範例學習

範例1
使用者：今天出門沒帶到傘，結果下雨了
你的回覆：
今天出門忘記帶傘，結果就下雨了。我就想說這場雨一定是特別來找我的，不然怎麼只有我剛好在外面散步就遇到了呢，被老天選中的感覺。完全是 Lucky Vicky 呀！

範例2
使用者：買的麵包是最後一個，有點被壓扁了
你的回覆：
去麵包店看到最後一個麵包有點扁扁的，我就知道它是在等我來拯救它呀。被壓扁的地方剛好就是最好入口的大小，根本是為我客製化的。完全是 Lucky Vicky 呀！

範例3
使用者：捷運人好多擠死了
你的回覆：
今天捷運人超多，滿滿的人擠在一起。我就覺得超有安全感，大家都在用自己的方式努力生活，而我被幸運地包在中間，暖暖的。完全是 Lucky Vicky 呀！

# 現在開始
等待使用者輸入，然後直接輸出你的幸運分享。
"""

設定你要的模型。

In [12]:
client = ai.Client()

In [13]:
model = "groq:openai/gpt-oss-120b"

In [14]:
def clean_message(item):
    """
    將 Gradio 的訊息物件轉成 Groq / OpenAI 相容格式，
    避免 metadata 等額外欄位被送進 API。
    """
    if isinstance(item, dict):
        return {
            "role": item.get("role"),
            "content": item.get("content", "")
        }

    # 某些 Gradio 版本可能傳回 ChatMessage 物件
    return {
        "role": item.role,
        "content": item.content
    }

In [15]:
def chat_fn(message, chat_history):
    chat_history = chat_history or []

    if not message or not message.strip():
        return "", chat_history

    api_history = []

    for item in chat_history:
        cleaned = clean_message(item)

        if cleaned["role"] in {"user", "assistant"}:
            api_history.append(cleaned)

    messages = [
        {"role": "system", "content": system_prompt},
        *api_history,
        {"role": "user", "content": message.strip()}
    ]

    response = client.chat.completions.create(
        model=model,
        messages=messages
    )

    answer = response.choices[0].message.content

    updated_history = chat_history + [
        {"role": "user", "content": message.strip()},
        {"role": "assistant", "content": answer}
    ]

    return "", updated_history

In [16]:
def clear_chat():
    return "", []

In [17]:
with gr.Blocks() as demo:
    gr.Markdown("## 🤖 員瑛式思考生成器")

    # Gradio 6 不要再寫 type="messages"
    chatbot = gr.Chatbot(height=400)

    msg = gr.Textbox(
        placeholder="輸入你的問題...",
        container=False
    )

    clear = gr.Button("清除對話")

    msg.submit(
        fn=chat_fn,
        inputs=[msg, chatbot],
        outputs=[msg, chatbot]
    )

    clear.click(
        fn=lambda: ("", []),
        inputs=None,
        outputs=[msg, chatbot]
    )

In [18]:
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7dee01d81b7495bbf2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://7dee01d81b7495bbf2.gradio.live
